In [ ]:
import optuna
from torch.optim import Adam
from torch.optim.lr_scheduler import ExponentialLR
from torch.amp import GradScaler
from torch.utils.data import Subset
import random
import subprocess
subprocess.Popen(["optuna-dashboard", "sqlite:///optuna.db", "--port", "8080"])

study = optuna.create_study(
    direction="maximize",
    study_name="codebert_lr_search",
    storage="sqlite:///optuna.db",  # persiste los trials
    load_if_exists=True
)

In [ ]:
import optuna
from torch.optim import Adam
from torch.optim.lr_scheduler import ExponentialLR
from torch.amp import GradScaler
from torch.utils.data import Subset
import random
print("\nCargando tokenizador CodeBERT...")
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

train_ds_full = VulnDataset(TRAIN_CSV, tokenizer, MAX_LEN)
test_ds       = VulnDataset(TEST_CSV,  tokenizer, MAX_LEN)
    
    # Split 90/10 para validation
generator  = torch.Generator().manual_seed(42)
train_size = int(0.9 * len(train_ds_full)) # TODO: Modificar para optimizar usando todo el train set
val_size   = len(train_ds_full) - train_size
train_ds, val_ds = random_split(train_ds_full, [train_size, val_size], generator=generator)
    
print(f"  Train : {len(train_ds):,} funciones")
print(f"  Val   : {len(val_ds):,} funciones")
print(f"  Test  : {len(test_ds):,} funciones")
    
    # Distribución del training set (sobre el CSV completo, referencial)
train_df = pd.read_csv(TRAIN_CSV)
print("\n  Distribución train:")
for cls, n in train_df["vulnerability"].value_counts().items():
    print(f"    {cls:<28}: {n:>6,}  ({100 * n / len(train_df):.1f}%)")
    
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
def get_subset_loader(dataset, fraction=0.3, batch_size=BATCH_SIZE, shuffle=True):
    n = int(len(dataset) * fraction)
    indices = random.sample(range(len(dataset)), n)
    return DataLoader(
        Subset(dataset, indices),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=4,
        pin_memory=True,
    )


def objective(trial: optuna.Trial):
    lr_encoder = trial.suggest_float("lr_encoder", 1e-6, 8e-6, log=True) # TODO: Modificar para optimizar lr_encoder de capa de clasificación
    beta_1      = 0.91

    model = OptimizedCodeBERT(num_labels=NUM_LABELS, dropout=DROPOUT).to(DEVICE)
    ckpt  = torch.load(os.path.join(OUTPUT_DIR, "best_model.pt"), map_location=DEVICE)
    model.load_state_dict(ckpt["state_dict"])

    optimizer = Adam(
        [
            {"params": model.encoder.parameters(),    "lr": lr_encoder},
            {"params": model.classifier.parameters(), "lr": LR_HEAD},
        ],
        betas=(beta_1, 0.999),
        weight_decay=WEIGHT_DECAY,
    )
    class_weights = compute_class_weights(train_ds, 4, DEVICE)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    scaler    = GradScaler("cuda")

    # Subsets para optimizar tiempos, me voy a quedar sin tiempo de procesamiento en kaggle
    trial_train_loader = get_subset_loader(train_ds, fraction=0.3)
    trial_val_loader   = get_subset_loader(val_ds,   fraction=0.5, shuffle=False)

    for epoch in range(2):
        train_epoch(model, trial_train_loader, optimizer, criterion, scaler, DEVICE, GRAD_ACCUM)

    val_metrics, _, _, _ = evaluate(model, trial_val_loader, criterion, DEVICE)
    
    if NUM_GPUS > 1:
        model = nn.DataParallel(model)

    final_metrics, labels_np, preds_np, probs_np = evaluate(
            model, test_loader, criterion, DEVICE
    )
    
    print("\nMétricas globales:")
    print(f"  F1  macro  : {final_metrics['f1_macro']:.4f}")
    print(f"  F1  micro  : {final_metrics['f1_micro']:.4f}")
    print(f"  Recall mac : {final_metrics['recall_macro']:.4f}")
    print(f"  Precision  : {final_metrics['precision_macro']:.4f}")
    print(f"  Accuracy   : {final_metrics['accuracy']:.4f}")
    
    print_per_class_metrics(labels_np, preds_np)
    
    return val_metrics["f1_macro"]


study.optimize(objective, n_trials=10)